In [5]:
import os, random, shutil

In [9]:
original_dir = "../../data/raw/PetImages"
train_dir = "../../data/processed/Petimagespro/train"
val_dir = "../../data/processed/Petimagespro/val"

In [ ]:
for dir_path in [train_dir, val_dir]:
    for category in ["Cat", "Dog"]:
        os.makedirs(f"{dir_path}/{category.lower()}", exist_ok=True)

for category in ["Cat", "Dog"]:
    category_lower = category.lower()
    files = os.listdir(f"{original_dir}/{category}")
    files = [f for f in files if f.lower().endswith((".jpg", ".png"))]
    random.shuffle(files)
    split = int(0.8 * len(files))
    train_files = files[:split]
    val_files = files[split:]

    # copy ไฟล์ไป train
    for f in train_files:
        shutil.copy(f"{original_dir}/{category}/{f}", f"{train_dir}/{category_lower}/{f}")
    # copy ไฟล์ไป val
    for f in val_files:
        shutil.copy(f"{original_dir}/{category}/{f}", f"{val_dir}/{category_lower}/{f}")

In [ ]:
import torch.nn as nn
import torch
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms , datasets
from torch.utils.data import DataLoader



In [10]:
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.resize((128, 128))),  # ใช้ Lambda แทน Resize
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [15]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32*32*32, 128)
        self.fc2 = nn.Linear(128, 2)  # Binary classification: 2 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32*32*32)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()   # สำหรับ classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)

In [ ]:

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

NameError: name 'nn' is not defined

In [11]:
torch.save(model.state_dict(), "catordog_nn.pth")

NameError: name 'model' is not defined

In [16]:
model = SimpleCNN()
model.load_state_dict(torch.load("catordog_nn.pth"))
model.eval()
model.to(device)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)

In [18]:
from PIL import Image

img = Image.open("dog-hero.jpg").convert("RGB")
img = transform(img).unsqueeze(0)  # เพิ่ม batch dimension
img = img.to(device)

with torch.no_grad():
    output = model(img)
    _, predicted = torch.max(output, 1)

print(predicted.item())

class_names = train_dataset.classes
print(class_names[predicted.item()])

1
dog
